# Módulo 2 — Sentimiento: Baseline TF-IDF
**TF-IDF + Logistic Regression** sobre Amazon Reviews Multilingual (es)

> No requiere GPU. CPU es suficiente para este notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/smartretail360'
MODEL_DIR  = f'{DRIVE_ROOT}/models/sentiment_analyzer'

import os
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
!pip install -q datasets scikit-learn joblib

In [ ]:
from datasets import load_dataset
ds = load_dataset('amazon_reviews_multi', 'es')
print(ds)

In [ ]:
import re, unicodedata

def clean_text(text):
    text = unicodedata.normalize('NFKD', text).lower().strip()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\sáéíóúüñ]', ' ', text)
    return re.sub(r'\s+', ' ', text)

def rating_to_label(r):
    return 0 if r <= 2 else (1 if r == 3 else 2)

texts_train  = [clean_text(r['review_body']) for r in ds['train']]
labels_train = [rating_to_label(r['stars'])  for r in ds['train']]
texts_test   = [clean_text(r['review_body']) for r in ds['test']]
labels_test  = [rating_to_label(r['stars'])  for r in ds['test']]

print(f'Train: {len(texts_train)} | Test: {len(texts_test)}')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',  LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')),
])

pipeline.fit(texts_train, labels_train)
preds = pipeline.predict(texts_test)
print(classification_report(labels_test, preds, target_names=['negative', 'neutral', 'positive']))

In [ ]:
joblib.dump(pipeline, f'{MODEL_DIR}/tfidf_baseline.pkl')
print('Pipeline guardado en Drive.')